# 02 — Typed Events & Structured Output

lionag2's agents don't just return text — they emit **typed events** that drive reactive coordination. This tutorial introduces the event system and `response_schema` for structured output.

Set `OPENAI_API_KEY` in your `.env`.

In [1]:
import os

from dotenv import load_dotenv

load_dotenv()

from autogen.beta import Agent
from autogen.beta.config import OpenAIConfig

config = OpenAIConfig("gpt-5.4-mini", api_key=os.getenv("OPENAI_API_KEY"))

## lionag2's event types

Five `BaseEvent` subclasses carry typed research signals between agents and the coordination bus:

| Event | Purpose |
|---|---|
| `FindingEmitted` | A source-backed claim with novelty + confidence scores |
| `DepthRequested` | Request to spawn a child investigation |
| `ContradictionFound` | Conflicting claims across branches |
| `PivotDetected` | Evidence contradicted the initial hypothesis |
| `PaperGapEvent` | Paper writer identified a gap needing more research |

In [2]:
from lionag2.research.events import (
    FindingEmitted,
)

# Events are Pydantic-validated BaseEvent subclasses
f = FindingEmitted(
    claim="Spin-fluctuation pairing dominates near optimal doping",
    evidence="INS shows resonance at ~5*kB*Tc",
    source_agent="theorist",
    novelty=0.85,
    confidence=0.7,
    depth=0,
)
print(f"FindingEmitted: claim={f.claim!r}, novelty={f.novelty}, depth={f.depth}")

FindingEmitted: claim='Spin-fluctuation pairing dominates near optimal doping', novelty=0.85, depth=0


## Emission tools

Agents emit events via `@tool`-decorated functions. These are the **only** custom tools in lionag2 — everything else (search, code, memory) comes from AG2 native toolkits.

When an agent calls `emit_finding(claim=..., novelty=0.8)`, the tool creates a `FindingEmitted` event and sends it via `ctx.send()`. Observers on the agent pick it up reactively.

In [3]:
from lionag2.research.tools import EMISSION_TOOLS

for t in EMISSION_TOOLS:
    print(f"  {t.name}: {t.schema.function.description}")

  emit_finding: Emit a typed finding when you discover a source-backed claim.
  request_depth: Request a child research node for a high-value follow-up.
  emit_contradiction: Flag conflicting claims across branches.
  emit_pivot: Flag that evidence contradicted the initial hypothesis.
  handoff: Hand off to another specialist when you've contributed what you can.

    Pass 'done' as next_agent to end the team discussion.
    Available specialists are listed in your system prompt.
    


## Structured output with `response_schema`

For agents that need to return structured data (like the cross-checker or paper writer), AG2 supports `response_schema` with Pydantic models.

`Field(default=...)` and `Field(default_factory=...)` work at the Pydantic level — the model validates and fills defaults correctly. However, AG2 sends `strict: True` to OpenAI, and OpenAI's strict mode requires ALL properties in `required` (even those with defaults). Pydantic excludes default fields from `required`, so the schema gets rejected.

**`PromptedSchema`** works around this: it injects the JSON schema into the system prompt instead of using `response_format`, bypassing OpenAI's strict validation. Use it when your models have optional fields with defaults.

AG2 also supports **callable result** via the `@response_schema` decorator — it generates JSON schema from function arguments, then lets you execute arbitrary code to map the LLM answer to complex Python objects or add side effects. See [AG2 docs](https://docs.ag2.ai/latest/docs/beta/structured_output/#custom-validation-with-response_schema).

In [ ]:
from autogen.beta import PromptedSchema

from lionag2.research.models import CrossCheckReport

# PromptedSchema needed because CrossCheckReport has optional fields
# and AG2 uses strict: True with OpenAI's response_format
checker = Agent(
    "cross_checker",
    prompt="Cross-check these findings for contradictions and gaps.",
    config=config,
    response_schema=PromptedSchema(CrossCheckReport),
)

reply = await checker.ask(
    "Findings:\n"
    "- [theorist d=0] Spin fluctuations dominate pairing\n"
    "- [analyst d=1] Phonon contribution non-negligible at overdoping\n"
)

report = await reply.content(retries=1)
if report:
    print(f"Contradictions: {len(report.contradictions)}")
    print(f"Gaps: {len(report.gaps)}")
    print(f"Summary: {report.summary[:200]}")

BadRequestError: Error code: 400 - {'error': {'message': "Invalid schema for response_format 'CrossCheckReport': In context=(), 'required' is required to be supplied and to be an array including every key in properties. Missing 'source_a'.", 'type': 'invalid_request_error', 'param': 'response_format', 'code': None}}

## Up next

Exa search can return raw HTML that blows context windows. Tutorial 03 shows how AG2's `ToolMiddleware` cleans results before they hit the stream.